# Notebook 06 — Predictive Demand Modeling
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 5 — Modelagem Preditiva

Modelo preditivo de demanda por estação em janelas de 30min, com horizontes
t+1 (30min ahead) e t+2 (1h ahead). Saída alimenta o ROI engine (Integrante 6).

**Pipeline:**
1. Setup + decisões fechadas
2. Leitura Gold (demand_30min, dim_stations, station_hourly_profile)
3. Feature engineering (lags + perfis históricos)
4. Train/val split temporal
5. Baselines obrigatórios (Naive1, Naive2, Naive3)
6. Modelo principal — Spark MLlib GBTRegressor com CrossValidator
7. Avaliação (MAE/RMSE primário, F1 do alerta secundário)
8. Derivação do alerta (threshold calibrado no val set)
9. Persistência em `dados/gold/predictions/` + salvamento do pipeline MLlib
10. Inventário final


## 0. Decisões técnicas fechadas

Decisões fundamentadas em análise quantitativa do Silver/Gold em DEV (~3,9M trips, Dez/2025+Jan/2026).
Documentação completa: ver discussão do PR de Silver fix.


In [1]:
# ============================================================
# DECISÕES TÉCNICAS FECHADAS — ver doc do PR
# ============================================================
GRANULARITY_MIN     = 30                    # decisão 1 (CV=0,97 vs 0,78@15 e 1,17@1h)
HORIZONS            = [1, 2]                # decisão 2 (t+1=30min, t+2=1h)
FORMULATION         = 'regression_multi'    # decisão 3 (CRITICAL é 0,07% — classif. inviável)
MODEL_TYPE          = 'spark_mllib_gbt'     # decisão 4 (paradigma distribuído)
LAG_FEATURES        = [1, 2, 4]             # decisão 5 (lag-48/336 são ruído em DEV)
VALIDATION_STRATEGY = 'temporal_holdout'    # decisão 6 (50/12 dias)
TRAIN_DATE_END      = '2026-01-20'          # exclusive
VAL_DATE_END        = '2026-02-01'          # exclusive
PRIMARY_METRIC      = 'MAE'                 # decisão 7 (departures + arrivals separados)
ALERT_METRIC        = 'F1_critical'         # secundária (AUC-PR, confusion)
USE_MLFLOW          = True                  # decisão 8
TUNING_GRID = {                             # decisão 9 (4 combos × 3 folds)
    'maxDepth': [5, 8],
    'maxIter':  [50, 100],
}

# Premissas a re-validar em PROD:
# - lag-48 (1d) e lag-336 (1sem) podem virar úteis com 8 meses de histórico.
# - Threshold de alerta calibrado em DEV pode precisar recalibrar em PROD.
# - GBTRegressor pode não escalar com 23M linhas — fallback: SynapseML LightGBM.

print(f'Granularidade: {GRANULARITY_MIN}min')
print(f'Horizontes:    {HORIZONS} ({", ".join(str(h * GRANULARITY_MIN) + "min ahead" for h in HORIZONS)})')
print(f'Lags:          {LAG_FEATURES} ({", ".join(f"{l*GRANULARITY_MIN}min" for l in LAG_FEATURES)})')
print(f'Train end:     {TRAIN_DATE_END} (exclusive)')
print(f'Val end:       {VAL_DATE_END} (exclusive)')


Granularidade: 30min
Horizontes:    [1, 2] (30min ahead, 60min ahead)
Lags:          [1, 2, 4] (30min, 60min, 120min)
Train end:     2026-01-20 (exclusive)
Val end:       2026-02-01 (exclusive)


## 1. Setup & Dependências

In [2]:
import os
import sys
import platform
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

import mlflow
import mlflow.spark

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
print(f'pandas {pd.__version__}, numpy {np.__version__}, mlflow {mlflow.__version__}')


pandas 2.0.3, numpy 1.26.4, mlflow 3.11.1


In [3]:
# ============================================================
# Caminhos
# ============================================================
PROJECT_ROOT  = Path(os.getcwd()).parent
DATA_DIR      = PROJECT_ROOT / 'dados'
GOLD_DIR      = DATA_DIR / 'gold'
GOLD_PRED     = GOLD_DIR / 'predictions'
MODELS_DIR    = DATA_DIR / 'models'
MLFLOW_DIR    = DATA_DIR / 'mlruns'

GOLD_PRED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'GOLD_DIR:     {GOLD_DIR}')
print(f'GOLD_PRED:    {GOLD_PRED}')
print(f'MODELS_DIR:   {MODELS_DIR}')


PROJECT_ROOT: /home/jovyan/work
GOLD_DIR:     /home/jovyan/work/dados/gold
GOLD_PRED:    /home/jovyan/work/dados/gold/predictions
MODELS_DIR:   /home/jovyan/work/dados/models


In [4]:
# ============================================================
# SparkSession + Delta Lake
# ============================================================
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_active = SparkSession.getActiveSession()
if _active is not None:
    _active.stop()

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName('CitiBike_Int5_PredictiveModel')
    .config('spark.driver.memory', os.environ.get('SPARK_DRIVER_MEMORY', '8g'))
    .config('spark.sql.shuffle.partitions', '64')
    .config('spark.sql.session.timeZone', 'America/New_York')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.databricks.delta.retentionDurationCheck.enabled', 'false')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .master('local[*]')
).getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} + Delta Lake')
print(f'  Cores:         {spark.sparkContext.defaultParallelism}')
print(f'  Driver memory: {spark.conf.get("spark.driver.memory")}')


Spark 3.5.0 + Delta Lake
  Cores:         8
  Driver memory: 4g


In [5]:
# ============================================================
# MLflow setup (file-based tracking, sem servidor)
# ============================================================
mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
EXPERIMENT_NAME = 'citibike_demand_v1'
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment:          {EXPERIMENT_NAME}')


/opt/conda/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/11 19:48:06 INFO mlflow.tracking.fluent: Experiment with name 'citibike_demand_v1' does not exist. Creating a new experiment.


MLflow tracking URI: file:///home/jovyan/work/dados/mlruns
Experiment:          citibike_demand_v1


## 2. Leitura Gold

Tabelas consumidas:
- `demand_30min` — grão `(station_id, window_30min)`, alvos `departures` e `arrivals`.
- `dim_stations` — `popularity_tier` (alta/média/baixa).
- `station_hourly_profile` — média histórica por `(station_id, hour_of_day, day_of_week)` (feature poderosa).
- `agg_weather_impact` — contexto climático.


In [6]:
demand_30 = spark.read.format('delta').load(str(GOLD_DIR / 'demand_30min'))
dim_stations = spark.read.format('delta').load(str(GOLD_DIR / 'dim_stations'))
station_profile = spark.read.format('delta').load(str(GOLD_DIR / 'station_hourly_profile'))

print(f'demand_30min:           {demand_30.count():>10,} rows × {len(demand_30.columns)} cols')
print(f'dim_stations:           {dim_stations.count():>10,} rows × {len(dim_stations.columns)} cols')
print(f'station_hourly_profile: {station_profile.count():>10,} rows × {len(station_profile.columns)} cols')

print('\n=== Schema demand_30min ===')
demand_30.printSchema()


demand_30min:            1,716,812 rows × 12 cols
dim_stations:                2,407 rows × 15 cols
station_hourly_profile:    282,551 rows × 9 cols

=== Schema demand_30min ===
root
 |-- station_id: string (nullable = true)
 |-- window_30min: timestamp (nullable = true)
 |-- station_name: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_peak_hour: boolean (nullable = true)
 |-- departures: long (nullable = true)
 |-- peak_departures: long (nullable = true)
 |-- avg_trip_duration_min: double (nullable = true)
 |-- arrivals: long (nullable = true)
 |-- peak_arrivals: long (nullable = true)
 |-- net_flow: long (nullable = true)



## 3. Feature Engineering

Construído via Spark Window functions (sem UDFs Python — padrão do projeto).

**Features criadas:**
- **Lags:** `dep_lag1`, `dep_lag2`, `dep_lag4`, `arr_lag1`, `arr_lag2`, `arr_lag4`
- **Rolling stats:** média de departures nas últimas 4 janelas (2h)
- **Calendário:** `is_weekend`, `is_peak_hour` já vêm do Gold
- **Categóricas join:** `popularity_tier` (de dim_stations), perfil histórico (de station_hourly_profile)

**Targets:** `target_dep_t1`, `target_dep_t2`, `target_arr_t1`, `target_arr_t2` (lead 1 e 2).


In [7]:
# Janela temporal por estação (chave do partitionBy + ordering)
w_station = Window.partitionBy('station_id').orderBy('window_30min')

df_fe = demand_30.select(
    'station_id', 'station_name', 'window_30min',
    'hour_of_day', 'day_of_week', 'is_peak_hour',
    'departures', 'arrivals'
)

# Lags (features) — 1, 2, 4 janelas atrás
for lag in LAG_FEATURES:
    df_fe = (
        df_fe
        .withColumn(f'dep_lag{lag}', F.lag('departures', lag).over(w_station))
        .withColumn(f'arr_lag{lag}', F.lag('arrivals', lag).over(w_station))
    )

# Rolling stats (média das últimas 4 janelas, sem incluir a atual)
df_fe = (
    df_fe
    .withColumn('dep_rolling_avg_4',
                F.avg('departures').over(w_station.rowsBetween(-4, -1)))
    .withColumn('arr_rolling_avg_4',
                F.avg('arrivals').over(w_station.rowsBetween(-4, -1)))
)

# is_weekend (Spark dayofweek: 1=domingo, 7=sábado)
df_fe = df_fe.withColumn('is_weekend', F.col('day_of_week').isin([1, 7]).cast('int'))

# Targets (lead): t+1 e t+2 janelas à frente
for h in HORIZONS:
    df_fe = (
        df_fe
        .withColumn(f'target_dep_t{h}', F.lead('departures', h).over(w_station))
        .withColumn(f'target_arr_t{h}', F.lead('arrivals',   h).over(w_station))
    )

print(f'Após FE: {df_fe.count():,} rows × {len(df_fe.columns)} cols')
df_fe.printSchema()


Após FE: 1,716,812 rows × 21 cols
root
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- window_30min: timestamp (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_peak_hour: boolean (nullable = true)
 |-- departures: long (nullable = true)
 |-- arrivals: long (nullable = true)
 |-- dep_lag1: long (nullable = true)
 |-- arr_lag1: long (nullable = true)
 |-- dep_lag2: long (nullable = true)
 |-- arr_lag2: long (nullable = true)
 |-- dep_lag4: long (nullable = true)
 |-- arr_lag4: long (nullable = true)
 |-- dep_rolling_avg_4: double (nullable = true)
 |-- arr_rolling_avg_4: double (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- target_dep_t1: long (nullable = true)
 |-- target_arr_t1: long (nullable = true)
 |-- target_dep_t2: long (nullable = true)
 |-- target_arr_t2: long (nullable = true)



In [8]:
# Join com popularity_tier
dim_stations_slim = dim_stations.select(
    F.col('station_id').alias('dim_station_id'),
    'popularity_tier'
)
df_fe = df_fe.join(
    dim_stations_slim,
    df_fe.station_id == dim_stations_slim.dim_station_id,
    'left'
).drop('dim_station_id')

# Join com perfil histórico (avg_trips_per_day por hour_of_day × day_of_week)
profile_slim = station_profile.select(
    F.col('start_station_id').alias('p_station_id'),
    F.col('hour_of_day').alias('p_hour'),
    F.col('day_of_week').alias('p_dow'),
    F.col('avg_trips_per_day').alias('hist_avg_trips_per_day'),
    F.col('total_trips').alias('hist_total_trips'),
)
df_fe = df_fe.join(
    profile_slim,
    (df_fe.station_id == profile_slim.p_station_id) &
    (df_fe.hour_of_day == profile_slim.p_hour) &
    (df_fe.day_of_week == profile_slim.p_dow),
    'left'
).drop('p_station_id', 'p_hour', 'p_dow')

print(f'Após joins: {df_fe.count():,} rows × {len(df_fe.columns)} cols')


Após joins: 1,716,812 rows × 24 cols


In [9]:
# Drop linhas com NULL em features ou targets críticos
features_required = [f'dep_lag{l}' for l in LAG_FEATURES] + [f'arr_lag{l}' for l in LAG_FEATURES]
features_required += ['dep_rolling_avg_4', 'arr_rolling_avg_4']
targets_required = [f'target_dep_t{h}' for h in HORIZONS] + [f'target_arr_t{h}' for h in HORIZONS]

before = df_fe.count()
df_fe = df_fe.dropna(subset=features_required + targets_required)
after = df_fe.count()
print(f'Drop NaN (lags+targets): {before:,} -> {after:,} ({(before-after)/before*100:.2f}% removidas)')

# Imputação simples de popularity_tier e perfil histórico
df_fe = df_fe.fillna({
    'popularity_tier': 'unknown',
    'hist_avg_trips_per_day': 0.0,
    'hist_total_trips': 0,
})


Drop NaN (lags+targets): 1,716,812 -> 1,703,557 (0.77% removidas)


## 4. Train/Validation Split (Temporal)

- **Train:** `window_30min < TRAIN_DATE_END` (2025-12-01 → 2026-01-19, ~50 dias)
- **Val:**   `TRAIN_DATE_END <= window_30min < VAL_DATE_END` (2026-01-20 → 2026-01-31, ~12 dias)

NUNCA `randomSplit` — quebra estrutura temporal.


In [10]:
train_df = df_fe.filter(F.col('window_30min') < F.lit(TRAIN_DATE_END).cast('timestamp'))
val_df   = df_fe.filter(
    (F.col('window_30min') >= F.lit(TRAIN_DATE_END).cast('timestamp')) &
    (F.col('window_30min') <  F.lit(VAL_DATE_END).cast('timestamp'))
)

n_train = train_df.count()
n_val = val_df.count()
print(f'Train: {n_train:>10,} rows ({(n_train/(n_train+n_val))*100:.1f}%)')
print(f'Val:   {n_val:>10,} rows ({(n_val/(n_train+n_val))*100:.1f}%)')

# Range temporal de cada split
for name, df in [('Train', train_df), ('Val', val_df)]:
    r = df.agg(F.min('window_30min').alias('min'), F.max('window_30min').alias('max')).collect()[0]
    print(f'  {name}: {r["min"]} -> {r["max"]}')


Train:  1,483,199 rows (87.1%)
Val:      220,358 rows (12.9%)
  Train: 2025-12-01 07:00:00 -> 2026-01-20 04:30:00
  Val: 2026-01-20 05:00:00 -> 2026-02-01 03:30:00


## 5. Baselines (sanity check)

Modelo principal precisa BATER os 3 baselines em MAE.

- **Naive1 — Perfil histórico:** prever a média histórica de departures para `(hour_of_day, day_of_week)` da estação. Vem do `station_hourly_profile`.
- **Naive2 — Persistência t-1:** `predicted_t+1 = observed_t`. (lag-1 já no FE.)
- **Naive3 — Mesma janela ontem:** `predicted_t+1 = observed_t-48`. **Atenção:** com lag-48 sendo ruído (autocorr ≈ 0 em DEV), Naive3 aqui deve ser FRACO. Documentamos como contra-evidência.


In [11]:
def evaluate_baseline(predictions_df, target_col, pred_col, name, horizon):
    """Computa MAE e RMSE de um baseline."""
    eval_df = predictions_df.filter(
        F.col(target_col).isNotNull() & F.col(pred_col).isNotNull()
    ).withColumn(
        target_col, F.col(target_col).cast('double')
    ).withColumn(
        pred_col, F.col(pred_col).cast('double')
    )
    n = eval_df.count()
    if n == 0:
        return {'baseline': name, 'horizon': horizon, 'mae': None, 'rmse': None, 'n': 0}

    mae_eval = RegressionEvaluator(
        labelCol=target_col, predictionCol=pred_col, metricName='mae'
    )
    rmse_eval = RegressionEvaluator(
        labelCol=target_col, predictionCol=pred_col, metricName='rmse'
    )
    return {
        'baseline': name,
        'horizon':  horizon,
        'target':   target_col,
        'mae':      mae_eval.evaluate(eval_df),
        'rmse':     rmse_eval.evaluate(eval_df),
        'n':        n,
    }


In [12]:
# Naive1 — perfil histórico (hist_avg_trips_per_day já está em df_fe; é por DIA, vamos converter para janela de 30min)
# avg por janela de 30min ≈ avg_trips_per_day / 48
val_baselines = (
    val_df
    .withColumn('naive1_pred_dep', F.col('hist_avg_trips_per_day') / 48.0)
    .withColumn('naive1_pred_arr', F.col('hist_avg_trips_per_day') / 48.0)
    # Naive2 = persistência: prever t+1 = valor atual (lag-0 efetivo)
    .withColumn('naive2_pred_dep', F.col('departures').cast('double'))
    .withColumn('naive2_pred_arr', F.col('arrivals').cast('double'))
    # Naive3 = lag-1 (mesma janela 30min antes; aproximação de "persistência t-1")
    # NOTA: Naive3 ideal seria lag-48 (1 dia atrás), mas autocorr em DEV é ~0,
    # então usamos lag-1 como segundo baseline simples.
    .withColumn('naive3_pred_dep', F.col('dep_lag1').cast('double'))
    .withColumn('naive3_pred_arr', F.col('arr_lag1').cast('double'))
)

baseline_results = []
for h in HORIZONS:
    for which in ['dep', 'arr']:
        target = f'target_{which}_t{h}'
        for nv in ['naive1', 'naive2', 'naive3']:
            pred = f'{nv}_pred_{which}'
            baseline_results.append(evaluate_baseline(val_baselines, target, pred, nv, h))

baseline_df = pd.DataFrame(baseline_results)
print(baseline_df.to_string(index=False))


baseline  horizon        target      mae     rmse      n
  naive1        1 target_dep_t1 1.960128 2.750360 220358
  naive2        1 target_dep_t1 0.985047 1.761245 220358
  naive3        1 target_dep_t1 1.059444 1.927654 220358
  naive1        1 target_arr_t1 1.413506 2.571678 220358
  naive2        1 target_arr_t1 1.172601 1.904611 220358
  naive3        1 target_arr_t1 1.269988 2.117819 220358
  naive1        2 target_dep_t2 1.957877 2.752685 220358
  naive2        2 target_dep_t2 1.057311 1.925748 220358
  naive3        2 target_dep_t2 1.135657 2.099680 220358
  naive1        2 target_arr_t2 1.411633 2.570440 220358
  naive2        2 target_arr_t2 1.266643 2.112797 220358
  naive3        2 target_arr_t2 1.358930 2.294668 220358


## 6. Modelo Principal — Spark MLlib GBTRegressor

Treina **um modelo por target**: `target_dep_t1`, `target_dep_t2`, `target_arr_t1`, `target_arr_t2`.
Pipeline: StringIndexer (popularity_tier) → OneHotEncoder → VectorAssembler → GBTRegressor.
Tuning: `ParamGridBuilder` (4 combos) × `CrossValidator` (3 folds).

Cada modelo é logado no MLflow com hyperparams, métricas e artefato salvo.


In [13]:
def build_pipeline(label_col):
    # Categóricas
    idx_pop = StringIndexer(inputCol='popularity_tier', outputCol='popularity_idx',
                            handleInvalid='keep')
    ohe_pop = OneHotEncoder(inputCol='popularity_idx', outputCol='popularity_oh',
                            handleInvalid='keep')

    # Features numéricas
    feature_cols = [
        'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour',
        'departures', 'arrivals',
        'hist_avg_trips_per_day', 'hist_total_trips',
        'dep_rolling_avg_4', 'arr_rolling_avg_4',
    ]
    feature_cols += [f'dep_lag{l}' for l in LAG_FEATURES]
    feature_cols += [f'arr_lag{l}' for l in LAG_FEATURES]

    assembler = VectorAssembler(
        inputCols=feature_cols + ['popularity_oh'],
        outputCol='features',
        handleInvalid='keep',
    )

    gbt = GBTRegressor(
        featuresCol='features',
        labelCol=label_col,
        seed=42,
    )

    pipeline = Pipeline(stages=[idx_pop, ohe_pop, assembler, gbt])
    return pipeline, gbt


In [14]:
def train_target(target_col):
    """Treina pipeline para um target com hyperparams fixos. Retorna best model + métricas.
    Versão v1 sem CrossValidator (cabe em 4g driver).
    """
    pipeline, gbt = build_pipeline(target_col)
    # Hyperparams fixos — escolhidos com base em GBTRegressor defaults + ajuste conservador
    gbt.setMaxDepth(5)
    gbt.setMaxIter(50)

    with mlflow.start_run(run_name=f'gbt_{target_col}', nested=False):
        mlflow.log_params({
            'target': target_col,
            'lags': str(LAG_FEATURES),
            'maxDepth': 5,
            'maxIter':  50,
            'tuning': 'fixed',
            'train_rows': train_df.count(),
            'val_rows':   val_df.count(),
        })

        logger.info(f'Treinando {target_col} (maxDepth=5, maxIter=50)...')
        model = pipeline.fit(train_df)

        # Avaliação no val
        preds = model.transform(val_df)
        mae = RegressionEvaluator(labelCol=target_col, predictionCol='prediction',
                                  metricName='mae').evaluate(preds)
        rmse = RegressionEvaluator(labelCol=target_col, predictionCol='prediction',
                                   metricName='rmse').evaluate(preds)
        r2 = RegressionEvaluator(labelCol=target_col, predictionCol='prediction',
                                 metricName='r2').evaluate(preds)

        mlflow.log_metrics({'val_mae': mae, 'val_rmse': rmse, 'val_r2': r2})

        # Salvar pipeline em disco
        model_path = MODELS_DIR / f'gbt_{target_col}_v1'
        if model_path.exists():
            import shutil
            shutil.rmtree(model_path)
        model.write().overwrite().save(str(model_path))
        mlflow.log_artifact(str(model_path), artifact_path=f'pipeline_{target_col}')

        logger.info(f'  {target_col}: MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')
        return {
            'target': target_col,
            'best_maxDepth': 5,
            'best_maxIter':  50,
            'val_mae':  mae,
            'val_rmse': rmse,
            'val_r2':   r2,
            'model':    model,
        }


In [15]:
# Treinar 4 modelos (2 horizontes × 2 targets)
target_cols = [f'target_dep_t{h}' for h in HORIZONS] + [f'target_arr_t{h}' for h in HORIZONS]
results = []
for tc in target_cols:
    results.append(train_target(tc))

# Resumo
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in results])
print('\n=== Resumo dos modelos ===')
print(results_df.to_string(index=False))


2026-05-11 19:50:27,762 [INFO] Treinando target_dep_t1 (maxDepth=5, maxIter=50)...
2026-05-11 19:53:21,299 [INFO]   target_dep_t1: MAE=0.898  RMSE=1.373  R²=0.515
2026-05-11 19:53:25,299 [INFO] Treinando target_dep_t2 (maxDepth=5, maxIter=50)...
2026-05-11 19:55:50,046 [INFO]   target_dep_t2: MAE=0.938  RMSE=1.442  R²=0.465
2026-05-11 19:55:53,715 [INFO] Treinando target_arr_t1 (maxDepth=5, maxIter=50)...
2026-05-11 19:58:11,322 [INFO]   target_arr_t1: MAE=1.047  RMSE=1.541  R²=0.516
2026-05-11 19:58:14,357 [INFO] Treinando target_arr_t2 (maxDepth=5, maxIter=50)...
2026-05-11 20:00:29,777 [INFO]   target_arr_t2: MAE=1.095  RMSE=1.625  R²=0.460



=== Resumo dos modelos ===
       target  best_maxDepth  best_maxIter  val_mae  val_rmse   val_r2
target_dep_t1              5            50 0.898301  1.373282 0.514840
target_dep_t2              5            50 0.938138  1.441878 0.464708
target_arr_t1              5            50 1.046616  1.541255 0.515541
target_arr_t2              5            50 1.095355  1.624889 0.460180


## 7. Avaliação — Modelo principal vs baselines

Critério: modelo principal precisa bater Naive1/2/3 em MAE no val set.


In [16]:
# Comparar modelo principal vs baselines (MAE)
gbt_results = []
for r in results:
    gbt_results.append({
        'baseline': 'GBT',
        'horizon': int(r['target'].split('_t')[1]),
        'target':  r['target'],
        'mae':     r['val_mae'],
        'rmse':    r['val_rmse'],
        'n':       n_val,
    })

all_results = pd.concat([baseline_df, pd.DataFrame(gbt_results)], ignore_index=True)
pivot = all_results.pivot_table(
    index=['target', 'horizon'], columns='baseline', values='mae'
).round(3)
print('=== MAE comparison (val set) ===')
print(pivot.to_string())

# Diferença GBT - melhor baseline
def best_baseline_mae(row):
    return min(v for k, v in row.items() if k != 'GBT' and pd.notna(v))

print('\n=== Ganho GBT vs melhor baseline (MAE) ===')
for (target, horizon), row in pivot.iterrows():
    bb = best_baseline_mae(row)
    gbt = row['GBT']
    diff = bb - gbt
    diff_pct = diff / bb * 100
    sign = 'BATEU' if diff > 0 else 'PIOR  '
    print(f'  {target} (h={horizon}): GBT={gbt:.3f}  best_baseline={bb:.3f}  diff={diff:+.3f} ({diff_pct:+.1f}%)  [{sign}]')


=== MAE comparison (val set) ===
baseline                 GBT  naive1  naive2  naive3
target        horizon                               
target_arr_t1 1        1.047   1.414   1.173   1.270
target_arr_t2 2        1.095   1.412   1.267   1.359
target_dep_t1 1        0.898   1.960   0.985   1.059
target_dep_t2 2        0.938   1.958   1.057   1.136

=== Ganho GBT vs melhor baseline (MAE) ===
  target_arr_t1 (h=1): GBT=1.047  best_baseline=1.173  diff=+0.126 (+10.7%)  [BATEU]
  target_arr_t2 (h=2): GBT=1.095  best_baseline=1.267  diff=+0.172 (+13.6%)  [BATEU]
  target_dep_t1 (h=1): GBT=0.898  best_baseline=0.985  diff=+0.087 (+8.8%)  [BATEU]
  target_dep_t2 (h=2): GBT=0.938  best_baseline=1.057  diff=+0.119 (+11.3%)  [BATEU]


In [17]:
# Métricas segmentadas — MAE por popularity_tier (para target_dep_t1 só, ilustração)
target_demo = 'target_dep_t1'
best_model_demo = next(r['model'] for r in results if r['target'] == target_demo)
val_preds = best_model_demo.transform(val_df).cache()

# Por popularity_tier
val_preds_with_err = val_preds.withColumn(
    'abs_err', F.abs(F.col('prediction') - F.col(target_demo))
)
mae_by_tier = (
    val_preds_with_err
    .groupBy('popularity_tier')
    .agg(F.avg('abs_err').alias('MAE'), F.count('*').alias('n'))
    .orderBy('popularity_tier')
    .toPandas()
)
print('=== MAE por popularity_tier (target_dep_t1) ===')
print(mae_by_tier.to_string(index=False))

# Por hour_of_day
mae_by_hour = (
    val_preds_with_err
    .groupBy('hour_of_day')
    .agg(F.avg('abs_err').alias('MAE'), F.count('*').alias('n'))
    .orderBy('hour_of_day')
    .toPandas()
)
print('\n=== MAE por hour_of_day (target_dep_t1) ===')
print(mae_by_hour.to_string(index=False))

val_preds.unpersist()


=== MAE por popularity_tier (target_dep_t1) ===
popularity_tier      MAE      n
           High 1.169161 134509
            Low 0.175970   4117
         Medium 0.485757  80704
        unknown 0.737492   1028

=== MAE por hour_of_day (target_dep_t1) ===
 hour_of_day      MAE     n
           0 0.791815  3658
           1 0.980939  7267
           2 1.229390 11409
           3 1.175738 13986
           4 0.890332 12752
           5 0.810211 11073
           6 0.831126 11261
           7 0.829000 11754
           8 0.847550 12480
           9 0.885884 13367
          10 0.963487 13949
          11 1.136136 14724
          12 1.108249 15296
          13 0.983827 14421
          14 0.838395 12509
          15 0.711248 10185
          16 0.663838  8180
          17 0.594215  6775
          18 0.502820  5167
          19 0.446080  3928
          20 0.438147  2272
          21 0.396687  1453
          22 0.432148  1040
          23 0.508739  1452


DataFrame[station_id: string, station_name: string, window_30min: timestamp, hour_of_day: int, day_of_week: int, is_peak_hour: boolean, departures: bigint, arrivals: bigint, dep_lag1: bigint, arr_lag1: bigint, dep_lag2: bigint, arr_lag2: bigint, dep_lag4: bigint, arr_lag4: bigint, dep_rolling_avg_4: double, arr_rolling_avg_4: double, is_weekend: int, target_dep_t1: bigint, target_arr_t1: bigint, target_dep_t2: bigint, target_arr_t2: bigint, popularity_tier: string, hist_avg_trips_per_day: double, hist_total_trips: bigint, popularity_idx: double, popularity_oh: vector, features: vector, prediction: double]

## 8. Derivar alerta

`predicted_net_flow = predicted_arrivals - predicted_departures`.
Threshold calibrado no val set para maximizar F1 do CRITICAL.

CRITICAL: |net_flow| muito alto (estação esvaziando rápido ou superlotando).
WARNING: nível médio.
OK: padrão.


In [18]:
# Aplicar todos os modelos no val_df e construir DataFrame de predictions.
# Drop colunas intermediárias do pipeline entre iterações para evitar colisão.
def apply_all_models(base_df):
    PIPELINE_AUX_COLS = ['popularity_idx', 'popularity_oh', 'features']
    out = base_df
    for r in results:
        col_name = r['target'].replace('target_', 'pred_')
        m = r['model']
        # Drop auxiliares se existirem (de iteração anterior)
        for c in PIPELINE_AUX_COLS:
            if c in out.columns:
                out = out.drop(c)
        out = m.transform(out).withColumnRenamed('prediction', col_name)
    # Limpeza final
    for c in ['popularity_idx', 'popularity_oh', 'features']:
        if c in out.columns:
            out = out.drop(c)
    return out

val_with_preds = apply_all_models(val_df).cache()
val_with_preds.select('station_id', 'window_30min',
                      'pred_dep_t1', 'pred_dep_t2', 'pred_arr_t1', 'pred_arr_t2'
).show(5, truncate=False)

# Net flow previsto para t+1 (foco do alerta)
val_with_preds = (
    val_with_preds
    .withColumn('predicted_net_flow_t1',
                F.col('pred_arr_t1') - F.col('pred_dep_t1'))
    .withColumn('predicted_net_flow_t2',
                F.col('pred_arr_t2') - F.col('pred_dep_t2'))
)


+----------+-------------------+------------------+------------------+-------------------+-------------------+
|station_id|window_30min       |pred_dep_t1       |pred_dep_t2       |pred_arr_t1        |pred_arr_t2        |
+----------+-------------------+------------------+------------------+-------------------+-------------------+
|2086.07   |2026-01-20 06:30:00|1.1359876969962261|1.2241862194592998|0.18711687567396398|0.19145226311837463|
|2086.07   |2026-01-21 07:00:00|1.1629946818161627|1.019887301166704 |0.18086151162429528|0.11423722909712924|
|2086.07   |2026-01-21 15:00:00|1.0820324208233796|1.0687278299314007|0.21645775786691585|0.1559440217598426 |
|2086.07   |2026-01-22 15:00:00|1.1420439164807121|1.1101904497096244|0.30463246873578925|0.16922457299749158|
|2086.07   |2026-01-23 18:30:00|1.344588175231517 |1.1094127975721426|0.26582677630371454|0.09813199221089927|
+----------+-------------------+------------------+------------------+-------------------+-------------------+
o

In [19]:
# Calibração de threshold: para cada candidato de threshold absoluto,
# medir F1 do CRITICAL no val (binário: |net_flow| >= threshold = CRITICAL).
# Ground truth: CRITICAL ocorre quando |net_flow_observado_t+1| >= threshold_real.
# Usamos o quantil 99,9% do |net_flow| histórico como threshold de CRITICAL.

# Net flow observado em t+1 (já temos como (target_arr_t1 - target_dep_t1))
val_with_preds = val_with_preds.withColumn(
    'observed_net_flow_t1',
    F.col('target_arr_t1') - F.col('target_dep_t1')
)

# Threshold ground truth: top 0,1% absoluto
abs_threshold_gt = val_with_preds.approxQuantile(
    'observed_net_flow_t1', [0.99], 0.001
)[0]
abs_threshold_gt = abs(abs_threshold_gt)
print(f'Threshold CRITICAL (top 1% |net_flow| observado): {abs_threshold_gt:.2f}')

val_with_preds = val_with_preds.withColumn(
    'is_critical_observed',
    (F.abs(F.col('observed_net_flow_t1')) >= abs_threshold_gt).cast('int')
)

n_critical = val_with_preds.filter(F.col('is_critical_observed') == 1).count()
print(f'CRITICAL observados no val: {n_critical:,} de {n_val:,} ({n_critical/n_val*100:.3f}%)')


Threshold CRITICAL (top 1% |net_flow| observado): 6.00
CRITICAL observados no val: 5,574 de 220,358 (2.530%)


In [20]:
# Buscar threshold de predição que maximiza F1 do CRITICAL
candidates = [abs_threshold_gt * f for f in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5, 2.0]]

best = None
for thr in candidates:
    df_eval = val_with_preds.withColumn(
        'is_critical_pred',
        (F.abs(F.col('predicted_net_flow_t1')) >= thr).cast('int')
    )
    cm = df_eval.groupBy('is_critical_observed', 'is_critical_pred').count().toPandas()
    cm_dict = {(r['is_critical_observed'], r['is_critical_pred']): r['count'] for _, r in cm.iterrows()}
    tp = cm_dict.get((1, 1), 0)
    fp = cm_dict.get((0, 1), 0)
    fn = cm_dict.get((1, 0), 0)
    tn = cm_dict.get((0, 0), 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f'thr={thr:6.2f}  TP={tp:>5}  FP={fp:>5}  FN={fn:>5}  TN={tn:>7}  '
          f'P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}')
    if best is None or f1 > best['f1']:
        best = {'thr': thr, 'f1': f1, 'precision': precision, 'recall': recall,
                'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

print(f'\nMelhor threshold: {best["thr"]:.2f}  F1={best["f1"]:.3f}  '
      f'P={best["precision"]:.3f}  R={best["recall"]:.3f}')
ALERT_THRESHOLD = best['thr']


thr=  3.00  TP= 1512  FP= 2663  FN= 4062  TN= 212121  P=0.362  R=0.271  F1=0.310
thr=  3.60  TP= 1256  FP= 1687  FN= 4318  TN= 213097  P=0.427  R=0.225  F1=0.295
thr=  4.20  TP= 1051  FP= 1154  FN= 4523  TN= 213630  P=0.477  R=0.189  F1=0.270
thr=  4.80  TP=  855  FP=  763  FN= 4719  TN= 214021  P=0.528  R=0.153  F1=0.238
thr=  5.40  TP=  693  FP=  547  FN= 4881  TN= 214237  P=0.559  R=0.124  F1=0.203
thr=  6.00  TP=  539  FP=  357  FN= 5035  TN= 214427  P=0.602  R=0.097  F1=0.167
thr=  6.60  TP=  411  FP=  222  FN= 5163  TN= 214562  P=0.649  R=0.074  F1=0.132
thr=  7.20  TP=  322  FP=  149  FN= 5252  TN= 214635  P=0.684  R=0.058  F1=0.107
thr=  7.80  TP=  243  FP=   89  FN= 5331  TN= 214695  P=0.732  R=0.044  F1=0.082
thr=  9.00  TP=  124  FP=   38  FN= 5450  TN= 214746  P=0.765  R=0.022  F1=0.043
thr= 12.00  TP=   48  FP=    8  FN= 5526  TN= 214776  P=0.857  R=0.009  F1=0.017

Melhor threshold: 3.00  F1=0.310  P=0.362  R=0.271


## 9. Persistir Gold predictions

Schema combinado com Integrante 6 (Victor — ROI engine):
- Identificação: `station_id`, `window_30min`, `horizon`
- Predições: `predicted_departures`, `predicted_arrivals`, `predicted_net_flow`
- Alerta: `predicted_alert` (CRITICAL/OK)
- Confiança: `prediction_confidence` (placeholder; refinar em iteração futura)
- Metadados: `model_version`, `prediction_ts`

Particionado por `day_of_week` (padrão das tabelas Gold). OPTIMIZE + ZORDER em `(station_id, window_30min)`.


In [21]:
MODEL_VERSION = 'v1_dev_gbt'

# ===== Prediction confidence: 1 / (1 + MAE_segmento)
# Calcula MAE residual no val_df agregado por (station_id, hour_of_day, day_of_week)
# Confidence alta = segmento onde o modelo erra pouco; baixa = segmento ruidoso.
def build_confidence_table(target_col):
    eval_col = target_col.replace('target_', 'pred_')
    df = val_with_preds.select(
        'station_id', 'hour_of_day', 'day_of_week',
        F.abs(F.col(eval_col) - F.col(target_col)).alias('abs_err')
    )
    return (
        df.groupBy('station_id', 'hour_of_day', 'day_of_week')
        .agg(F.avg('abs_err').alias('mae_segment'))
    )

# Uma tabela de confidence por horizonte (foco em t+1 e t+2 dep, conservador)
conf_t1 = build_confidence_table('target_dep_t1').withColumnRenamed('mae_segment', 'mae_seg_t1')
conf_t2 = build_confidence_table('target_dep_t2').withColumnRenamed('mae_segment', 'mae_seg_t2')

# Construir DataFrame de predictions long-format (1 linha por (station, window, horizon))
def build_predictions_long(df_with_preds, horizon):
    mae_col = f'mae_seg_t{horizon}'
    conf_table = conf_t1 if horizon == 1 else conf_t2
    base = (
        df_with_preds
        .select(
            'station_id', 'station_name', 'window_30min',
            'hour_of_day', 'day_of_week',
            F.col(f'pred_dep_t{horizon}').alias('predicted_departures'),
            F.col(f'pred_arr_t{horizon}').alias('predicted_arrivals'),
            F.col(f'predicted_net_flow_t{horizon}').alias('predicted_net_flow'),
        )
        .withColumn('horizon', F.lit(horizon))
        .withColumn(
            'predicted_alert',
            F.when(F.abs(F.col('predicted_net_flow')) >= ALERT_THRESHOLD,
                   F.lit('CRITICAL')).otherwise(F.lit('OK'))
        )
    )
    # Join com a tabela de confidence (left → estações novas mantêm NULL)
    base = base.join(
        conf_table.withColumnRenamed('station_id', 'c_station_id')
                  .withColumnRenamed('hour_of_day', 'c_hour')
                  .withColumnRenamed('day_of_week', 'c_dow'),
        (base.station_id == F.col('c_station_id')) &
        (base.hour_of_day == F.col('c_hour')) &
        (base.day_of_week == F.col('c_dow')),
        'left'
    ).drop('c_station_id', 'c_hour', 'c_dow')
    # confidence = 1 / (1 + MAE_segmento). NULL se segmento sem histórico.
    base = base.withColumn(
        'prediction_confidence',
        F.when(F.col(mae_col).isNotNull(), 1.0 / (1.0 + F.col(mae_col)))
    ).drop(mae_col)
    base = base.withColumn('model_version', F.lit(MODEL_VERSION))
    base = base.withColumn('prediction_ts', F.current_timestamp())
    return base

predictions_t1 = build_predictions_long(val_with_preds, 1)
predictions_t2 = build_predictions_long(val_with_preds, 2)
predictions = predictions_t1.unionByName(predictions_t2)

print(f'predictions: {predictions.count():,} rows')
predictions.printSchema()
predictions.show(5, truncate=False)

# Sanity check da distribuição de confidence
conf_stats = predictions.agg(
    F.min('prediction_confidence').alias('min'),
    F.expr('percentile(prediction_confidence, 0.5)').alias('p50'),
    F.avg('prediction_confidence').alias('mean'),
    F.expr('percentile(prediction_confidence, 0.95)').alias('p95'),
    F.max('prediction_confidence').alias('max'),
    F.sum(F.col('prediction_confidence').isNull().cast('int')).alias('nulls'),
).collect()[0]
print(f'\nconfidence stats: min={conf_stats["min"]:.3f}  p50={conf_stats["p50"]:.3f}  mean={conf_stats["mean"]:.3f}  p95={conf_stats["p95"]:.3f}  max={conf_stats["max"]:.3f}  nulls={conf_stats["nulls"]:,}')


predictions: 440,716 rows
root
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- window_30min: timestamp (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- predicted_departures: double (nullable = false)
 |-- predicted_arrivals: double (nullable = false)
 |-- predicted_net_flow: double (nullable = false)
 |-- horizon: integer (nullable = false)
 |-- predicted_alert: string (nullable = false)
 |-- prediction_confidence: double (nullable = true)
 |-- model_version: string (nullable = false)
 |-- prediction_ts: timestamp (nullable = false)

+----------+-------------+-------------------+-----------+-----------+--------------------+-------------------+-------------------+-------+---------------+---------------------+-------------+--------------------------+
|station_id|station_name |window_30min       |hour_of_day|day_of_week|predicted_departures|predicted_arrivals |predicted_net_flow |horiz

In [22]:
# Escrever Delta particionado por day_of_week
(
    predictions.write
    .format('delta')
    .partitionBy('day_of_week')
    .mode('overwrite')
    .save(str(GOLD_PRED))
)

# OPTIMIZE + ZORDER
delta_pred = DeltaTable.forPath(spark, str(GOLD_PRED))
delta_pred.optimize().executeZOrderBy('station_id', 'window_30min')

print(f'predictions persistido em {GOLD_PRED}')
print('\n=== Delta history ===')
delta_pred.history(5).select('version', 'timestamp', 'operation').show(truncate=False)


predictions persistido em /home/jovyan/work/dados/gold/predictions

=== Delta history ===
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-05-11 16:00:59.544|OPTIMIZE |
|0      |2026-05-11 16:00:54.117|WRITE    |
+-------+-----------------------+---------+



## 10. Inventário final

In [23]:
# Tamanho em disco
def get_size(p):
    total = 0
    for f in Path(p).rglob('*'):
        if f.is_file():
            total += f.stat().st_size
    return total

pred_size = get_size(GOLD_PRED) / 1e6
n_pred = predictions.count()
n_critical_pred = predictions.filter(F.col('predicted_alert') == 'CRITICAL').count()

print(f'=== Inventário Notebook 06 ===')
print(f'predictions:      {n_pred:>10,} rows ({pred_size:.1f} MB)')
print(f'  CRITICAL:       {n_critical_pred:>10,} ({n_critical_pred/n_pred*100:.3f}%)')
print(f'  OK:             {n_pred - n_critical_pred:>10,}')
print(f'\nModelos salvos em {MODELS_DIR}:')
for d in sorted(MODELS_DIR.glob('gbt_*')):
    if d.is_dir():
        s = get_size(d) / 1e6
        print(f'  {d.name}  ({s:.1f} MB)')

print(f'\nMLflow runs em {MLFLOW_DIR}')
print(f'  Para visualizar: mlflow ui --backend-store-uri {MLFLOW_DIR}')


=== Inventário Notebook 06 ===
predictions:         440,716 rows (29.5 MB)
  CRITICAL:            7,431 (1.686%)
  OK:                433,285

Modelos salvos em /home/jovyan/work/dados/models:
  gbt_target_arr_t1_v1  (0.2 MB)
  gbt_target_arr_t2_v1  (0.2 MB)
  gbt_target_dep_t1_v1  (0.2 MB)
  gbt_target_dep_t2_v1  (0.2 MB)

MLflow runs em /home/jovyan/work/dados/mlruns
  Para visualizar: mlflow ui --backend-store-uri /home/jovyan/work/dados/mlruns


## 11. Limitações conhecidas

Premissas e restrições desta v1 do modelo. Importante revisar antes de qualquer uso operacional:

**Volume de dados.** Treino e validação rodaram em MODE=DEV, cobrindo apenas Dez/2025 e Jan/2026 (~3,9M viagens, 62 dias). PROD (8 meses, ~26,6M viagens) ainda não foi executado nesta sessão por restrição de RAM da máquina de desenvolvimento. Generalização para primavera/verão NYC não foi testada — o modelo foi treinado em janela de inverno e pode ter viés sazonal.

**Lags semanais (336 janelas).** A análise de autocorrelação em DEV mostrou lag-48 (1 dia) e lag-336 (1 semana) com correlação próxima de zero — foram descartados. Em PROD, com 8 meses, é plausível que voltem a ser úteis. Premissa a re-validar.

**Hyperparameter tuning desativado.** O CrossValidator (3 folds × 4 combinações de maxDepth/maxIter = 12 ajustes por modelo, 48 ajustes no total) estourou a RAM disponível (3.8 GiB no Docker Desktop). Modelo atual usa `maxDepth=5, maxIter=50` fixos — escolhidos por conservadorismo, sem garantia de ser o ótimo.

**Alerta CRITICAL é precision-orientado.** Threshold calibrado em 3 bikes/30min dá F1=0.31, Precision=0.36, Recall=0.27. O modelo perde ~73% dos eventos críticos reais. Para uso operacional, o ROI engine (Integrante 6) deve escolher um ponto da curva entre Precision e Recall conforme o custo de despachar caminhão sem necessidade vs. custo de deixar a estação esvaziar.

**Cold start de estações novas.** `prediction_confidence` é derivada do MAE residual por (station × hour × day_of_week) no val set. Estações sem histórico no `station_hourly_profile` (recém-instaladas) recebem confidence = NULL. Tratamento simples: o ROI engine deve fallback para média global ou descartar.

**Estatística da `popularity_tier`.** A categoria `unknown` cobre apenas 0,5% das predictions (1.028 linhas). Estações dessa categoria podem ter MAE pior por estarem fora da cadeia de joins do dim_stations — convém investigar caso o ROI dependa fortemente delas.


In [24]:
spark.stop()
print('SparkSession encerrada. Notebook 06 concluído.')


SparkSession encerrada. Notebook 06 concluído.
